# Plant disease classifier: train on Kaggle

Before you run:
1. **Settings -> Accelerator -> GPU** (T4 is fine).
2. **Settings -> Internet -> On** (needed to download pretrained weights and clone the repo).
3. **Add Data** -> search "PlantVillage" and add a dataset that has a `color/` folder with 38 class folders.

Run the cells top to bottom. Paste back the printed metrics and any error.

In [ ]:
!pip -q install onnx onnxruntime

In [ ]:
REPO_URL = "https://github.com/gangireddymayur/plant-disease-on-device.git"
!rm -rf /kaggle/working/repo && git clone -q $REPO_URL /kaggle/working/repo
!ls /kaggle/input

Set `DATA_DIR` to the dataset folder shown above (the one that contains `color/`, or the class folders directly).

In [ ]:
DATA_DIR = "/kaggle/input/plantvillage-dataset"  # <- change to match the listing above
!find $DATA_DIR -maxdepth 2 | head -20

## 1. Baseline: frozen backbone, train only the new head

In [ ]:
!cd /kaggle/working/repo && PYTHONPATH=src python -m plantdx.train \
    --data-dir $DATA_DIR --mode linear --epochs 3 --out-dir /kaggle/working/runs/linear

## 2. Fine-tune the whole network

In [ ]:
!cd /kaggle/working/repo && PYTHONPATH=src python -m plantdx.train \
    --data-dir $DATA_DIR --mode finetune --epochs 8 --out-dir /kaggle/working/runs/finetune

## 3. Compare the two runs

In [ ]:
import json
for name in ["linear", "finetune"]:
    m = json.load(open(f"/kaggle/working/runs/{name}/metrics.json"))
    print(name, "test:", m["test"], "| train seconds:", m["train_seconds"])

## 4. Export to ONNX and check it matches PyTorch

In [ ]:
!cd /kaggle/working/repo && PYTHONPATH=src python -m plantdx.export_onnx \
    --checkpoint /kaggle/working/runs/finetune/best.pt --out-dir /kaggle/working/model --quantize

## 5. Real-world check (important)
PlantVillage photos are taken on plain backgrounds, so the score above is optimistic. If you add a folder of real phone photos
(one sub-folder per class, using the same class names), measure the drop here:

In [ ]:
# !cd /kaggle/working/repo && PYTHONPATH=src python -m plantdx.evaluate \
#     --checkpoint /kaggle/working/runs/finetune/best.pt --data-dir /kaggle/input/YOUR_REAL_PHOTOS